# इमेज जनरेशन एप्लिकेशन बनाना (Azure OpenAI)

LLMs केवल टेक्स्ट जनरेशन तक सीमित नहीं हैं। आप टेक्स्ट विवरणों से इमेज भी जनरेट कर सकते हैं। इमेज एक माध्यम के रूप में मेडटेक, आर्किटेक्चर, पर्यटन, गेम डेवलपमेंट, मार्केटिंग, और भी कई क्षेत्रों में उपयोगी हैं। इस पाठ में हम Microsoft Foundry पर आज के **GPT इमेज** मॉडलों के साथ काम करते हैं।

## सीखने के लक्ष्य

इस पाठ के अंत तक आप सक्षम होंगे:

- समझाना कि इमेज जनरेशन क्या है और यह कहाँ उपयोगी है।
- `gpt-image` मॉडल परिवार को समझना और यह पुराने DALL·E मॉडलों से कैसे अलग है।
- एक इमेज जनरेशन एप्लिकेशन बनाना और इमेजेस को संपादित करना।

## इमेज जनरेशन क्या है?

इमेज जनरेशन मॉडल एक टेक्स्ट प्रॉम्प्ट से इमेज बनाते हैं। आधुनिक मॉडल जैसे `gpt-image` प्रशिक्षण के दौरान टेक्स्ट और इमेज के बीच के संबंध को सीखते हैं, फिर पुनरावृत्तिमूलक तरीके से यादृच्छिक शोर को आपकी प्रॉम्प्ट के अनुरूप इमेज में बदल देते हैं।

इमेज मॉडल के दो प्रसिद्ध परिवार हैं:

- **`gpt-image` (OpenAI)** — यह वर्तमान पीढ़ी है जो इस पाठ में उपयोग की जा रही है। यह टेक्स्ट-टू-इमेज जनरेशन और इमेज एडिटिंग (मास्क के साथ इनपेंटिंग) का समर्थन करता है।
- **Midjourney** — एक लोकप्रिय थर्ड-पार्टी मॉडल है जिसका अपना सेवा और Discord-आधारित कार्यप्रवाह है।

> पुराने OpenAI इमेज मॉडल — **DALL·E 2** और **DALL·E 3** — पुराने हैं। DALL·E 3 नई तैनाती के लिए उपलब्ध नहीं है, और `create_variation` जैसी विशेषताएं केवल DALL·E 2 में थीं। नए अनुप्रयोगों के लिए `gpt-image` मॉडल का उपयोग करें।

Microsoft Foundry पर, **`gpt-image-2`** नवीनतम और सबसे सक्षम इमेज मॉडल है और अनुशंसित डिफ़ॉल्ट है। `gpt-image-1.5` और `gpt-image-1-mini` भी आमतौर पर उपलब्ध हैं।

> **महत्वपूर्ण:** `gpt-image` मॉडल जनरेट की गई इमेज को **base64** (`b64_json`) में लौटाते हैं, URL के रूप में नहीं। आपका कोड base64 स्ट्रिंग को बाइट्स में डिकोड करता है और उसे सेव करता है — डाउनलोड करने के लिए कोई इमेज URL नहीं होता।


## अपनी पहली छवि जनरेशन एप्लिकेशन बनाना

तो छवि जनरेशन एप्लिकेशन बनाने के लिए क्या चाहिए? आपको निम्नलिखित लाइब्रेरी चाहिए:

- **python-dotenv**, इस लाइब्रेरी का उपयोग करने की अत्यधिक सिफारिश की जाती है ताकि आप अपने सीक्रेट्स को कोड से दूर *.env* फ़ाइल में रख सकें।
- **openai**, इस लाइब्रेरी का उपयोग आप OpenAI API के साथ बातचीत करने के लिए करेंगे।
- **pillow**, Python में छवियों के साथ काम करने के लिए।

यदि पहले से नहीं किया है, तो [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) पृष्ठ पर दिए गए निर्देशों का पालन करें ताकि Microsoft Foundry संसाधन और मॉडल बनाया जा सके। मॉडल के रूप में **gpt-image-2** चुनें (सबसे नवीनतम Azure OpenAI छवि मॉडल; DALL·E पुराना है)।

1. एक *.env* फ़ाइल बनाएं जिसमें निम्नलिखित सामग्री हो:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    अपने संसाधन के लिए Microsoft Foundry पोर्टल में "Deployments" अनुभाग में यह जानकारी देखें।



1. ऊपर दी गई लाइब्रेरीज़ को *requirements.txt* नामक फ़ाइल में इस तरह इकट्ठा करें:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. अगला, वर्चुअल एनवायरनमेंट बनाएँ और लाइब्रेरीज़ इंस्टॉल करें:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windows के लिए, अपना वर्चुअल वातावरण बनाने और सक्रिय करने के लिए निम्नलिखित कमांड्स का उपयोग करें:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* नामक फ़ाइल में निम्नलिखित कोड जोड़ें:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # dotenv आयात करें
    dotenv.load_dotenv()

    # Azure OpenAI (Microsoft Foundry) क्लाइंट कॉन्फ़िगर करें।
    # छवि मॉडलों के लिए हाल का API संस्करण आवश्यक है — अपने मॉडल के लिए आवश्यक संस्करण के लिए Foundry दस्तावेज़ देखिए।
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # छवि निर्माण API का उपयोग करके एक छवि बनाएं
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # अपनी प्रॉम्प्ट टेक्स्ट यहाँ दर्ज करें
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # उदाहरण के लिए "gpt-image-2"
        )
        # संग्रहीत छवि के लिए निर्देशिका सेट करें
        image_dir = os.path.join(os.curdir, 'images')

        # यदि निर्देशिका मौजूद नहीं है, तो इसे बनाएं
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # छवि पथ प्रारंभ करें (ध्यान दें कि फ़ाइल प्रकार png होना चाहिए)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image मॉडल छवि को base64 (b64_json) के रूप में वापस करते हैं, URL के रूप में नहीं
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # डिफ़ॉल्ट छवि दर्शक में छवि प्रदर्शित करें
        image = Image.open(image_path)
        image.show()

    # अपवाद पकड़ें
    except BadRequestError as err:
        print(err)

    ```

आइए इस कोड को समझते हैं:

- सबसे पहले, हम आवश्यक लाइब्रेरीज़ आयात करते हैं, जिनमें OpenAI लाइब्रेरी, dotenv लाइब्रेरी, base64 मॉड्यूल, और Pillow लाइब्रेरी शामिल हैं।

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- इसके बाद, हम *.env* फ़ाइल से पर्यावरण चर लोड करते हैं।

    ```python
    # dotenv इम्पोर्ट करें
    dotenv.load_dotenv()
    ```

- उसके बाद, हम Azure OpenAI (Microsoft Foundry) क्लाइंट को कॉन्फ़िगर करते हैं।

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- फिर, हम चित्र उत्पन्न करते हैं:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # अपना प्रॉम्प्ट टेक्स्ट यहाँ दर्ज करें
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` मॉडल इमेज को **base64** स्ट्रिंग के रूप में `data[0].b64_json` में लौटाते हैं। हम इसे बाइट्स में डिकोड कर फ़ाइल में लिखते हैं — डाउनलोड करने के लिए कोई URL नहीं होता।

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- अंत में, हम इमेज खोलते हैं और मानक इमेज व्यूअर का उपयोग करके उसे दिखाते हैं:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### चित्र उत्पन्न करने के और विवरण

आइए `images.generate` के पैरामीटर देखें:

- **prompt**, वह टेक्स्ट प्रॉम्प्ट है जिसका उपयोग चित्र उत्पन्न करने के लिए किया जाता है। यहां यह "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils" है।
- **size**, उत्पन्न चित्र का आकार है (उदाहरण के लिए `1024x1024`, `1536x1024`, `1024x1536`, या `"auto"`)।
- **n**, उत्पन्न चित्रों की संख्या है। यहां हम एक चित्र उत्पन्न करते हैं।
- **model**, आपका इमेज मॉडल डिप्लॉयमेंट नाम है (जैसे `gpt-image-2`)।

> इमेज मॉडल `temperature` पैरामीटर नहीं लेते — यह टेक्स्ट-जेनरेशन नियंत्रण है। विविधता पाने के लिए API को फिर से कॉल करें; विविधता कम करने के लिए, अपने प्रॉम्प्ट को अधिक विशिष्ट बनाएं।

## चित्र उत्पन्न करने की अतिरिक्त क्षमताएं

आपने देखा कि कुछ पंक्तियों के Python कोड से चित्र कैसे उत्पन्न करें। `gpt-image` मॉडल एक मौजूदा चित्र को **संपादित** भी कर सकते हैं। एक चित्र, वैकल्पिक **मास्क** (जो बदलाव के क्षेत्र को चिन्हित करता है), और एक प्रॉम्प्ट प्रदान करके, आप चित्र के किसी भाग को बदल सकते हैं — उदाहरण के लिए, हमारे बनी पर एक टोपी जोड़ना।

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# संपादनों को भी base64 के रूप में लौटाया जाता है
edited_image = base64.b64decode(response.data[0].b64_json)
```

मूल चित्र में केवल खरगोश है; अंतिम चित्र में खरगोश पर टोपी है।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
